In [ ]:
#VERSION 1.0 - JUST TO SEE IT AS EXAMPLE AND CLARFIY WHAT IS NEEDED
def _get_sgen_results(net, ppc):

    if "sgen" not in net or len(net.sgen) == 0:
        return

    # --- Map pandapower to ppc indices ---
    bus_lookup = net["_pd2ppc_lookups"]["bus"]
    sgen_df = net.sgen.loc[net.sgen.in_service]

    results = []

    for i, s in sgen_df.iterrows():
        bus = s.bus
        bus_ppc = bus_lookup[bus]
        vn_kv = net.bus.at[bus, "vn_kv"]
        base_z_ohm = (vn_kv ** 2) #/ ppc["baseMVA"]  # system base impedance [Ohm]

        gen_type = s.get("generator_type", "current_source")

        # --- Rated current ---
        Irated_ka = s.sn_mva / (np.sqrt(3) * vn_kv)

        # --- Extract local bus voltage (pre-fault) from ppc if available ---
        try:
            V_bus_pu = ppci_1["bus"][bus_ppc, VM] * vn_kv / np.sqrt(3)
            V_bus_kv=V_bus_pu* ppci_1["bus"][bus_ppc, BASE_KV]

        except Exception:
            V_bus_kv = vn_kv / np.sqrt(3)

        # --- Determine short-circuit current or impedance depending on type ---
        if gen_type == "current_source":
            # Full converter (PV, BESS)
            k_I = s.get("k", 1.2)
            phi_deg = -90  # lagging reactive
            I_ka = k_I * Irated_ka
            I_phasor = I_ka * np.exp(1j * np.deg2rad(phi_deg))
            S_mva = 3 * V_pre_kv * np.abs(I_ka)

        elif gen_type == "async":
            # Asynchronous generator / motor (IEC 60909-0 §6.10)
            '''lrc_pu = s.get("lrc_pu", 6.0)  # locked-rotor current multiple
            rx = s.get("rx", 7.0)
            Z_ohm = (vn_kv ** 2) / (s.sn_mva * lrc_pu)
            Z_complex = (rx + 1j) * Z_ohm / np.sqrt(1 + rx ** 2)'''
            sgen_gs = ppci_1['bus'][bus_idx, GS]
            sgen_bs = ppci_1['bus'][bus_idx, BS]
            y_g_pu = (sgen_gs+1j*sgen_bs)
            z_g_pu = 1/ y_g_pu
            z_g_ohm = z_g_pu * base_z_ohm
            
            I_ka = vn_kv / (np.sqrt(3) * np.abs(z_g_ohm))
            I_phasor = I_ka * np.exp(-1j * np.angle(z_g_ohm))
            S_mva = 3 * V_pre_kv * np.abs(I_ka)

        elif gen_type == "async_doubly_fed":
            # DFIG wind turbine (IEC 60909-0 §6.11)
            kappa = s.get("kappa", 1.1)
            rx = s.get("rx", 7.0)
            max_ik_ka = s.get("max_ik_ka", 1.2 * Irated_ka)
            sgen_gs = ppci_1['bus'][bus_idx, GS]
            sgen_bs = ppci_1['bus'][bus_idx, BS]
            y_g_pu = (sgen_gs+1j*sgen_bs)
            z_g_pu = 1 / y_g_pu
            z_g_ohm = z_g_pu * base_z_ohm
            
            I_ka = vn_kv / (np.sqrt(3) * np.abs(z_g_ohm))
            I_phasor = I_ka * np.exp(-1j * np.angle(z_g_ohm))
            S_mva = 3 * V_pre_kv * np.abs(I_ka)

        else:
            # Default to current-source model if unknown
            k_I = s.get("k", 1.2)
            phi_deg = -90
            I_ka = k_I * Irated_ka
            I_phasor = I_ka * np.exp(1j * np.deg2rad(phi_deg))
            S_mva = 3 * V_pre_kv * np.abs(I_ka)

        # --- Store result ---
        results.append({
            "bus": bus,
            "generator_type": gen_type,
            "vn_kv": vn_kv,
            "ikss_ka": np.abs(I_ka),
            "angle_deg": np.angle(I_phasor, deg=True),
            "skss_mva": S_mva,
            "p_mw": S_mva * np.cos(np.angle(I_phasor)) / 1e3,
            "q_mvar": S_mva * np.sin(np.angle(I_phasor)) / 1e3,
        })

    # --- Store in pandapower result table ---
    net.res_sgen_sc = pd.DataFrame(results)
    return net

In [1]:
#VERsion 1.2 - more logic used from currents.py but this is for all current contribution sources(all sgens) - not one
def _get_sgen_results(net, ppci_0, ppci_1, ppci_2, bus):
    if "sgen" not in net or len(net.sgen) == 0:
        return

    # --- Map pandapower to ppc indices ---
    bus_lookup = net["_pd2ppc_lookups"]["bus"]
    sgen_df = net.sgen.loc[net.sgen.in_service]

    results = []


    for i, s in sgen_df.iterrows():
        bus = s.bus
        bus_ppc = bus_lookup[bus]
        vn_sgen = net.bus.at[bus, "vn_kv"]
        base_z_ohm = (vn_sgen ** 2)  # / ppc["baseMVA"]  # system base impedance [Ohm]

        if net["_options"]["fault"] == "LLL":  # 3-phase fault
            #I1 = ppci_1["bus"][bus, IKSSC]
            #I0 = I2 = np.zeros_like(I1) works as well
            I1 = ppci_1["bus"][bus, IKSSC]
            I2 = ppci_2["bus"][bus, IKSSC]
            I0 = ppci_0["bus"][bus, IKSSC]

        elif net["_options"]["fault"] == "LG":  # single line-to-ground fault 7.5 standard
            #I0 = I1 = I2 = ppci_1['bus'][bus, IKSSC] / 3 this works as well as this bellow
            '''
            In LG faults, pandapower scales sequence currents (I0, I1, I2) by 3 in _calc_ikss_to_g().
            To get correct phase currents in _get_sgen_results(), divide each by 3 before using sequence_to_phase().
            Otherwise, Ia will be 3× too high due to internal scaling; Ib and Ic ≈ 0 as expected for single-phase faults.
            '''
            I1 = ppci_1["bus"][bus, IKSSC] / 3
            I2 = ppci_2["bus"][bus, IKSSC] / 3
            I0 = ppci_0["bus"][bus, IKSSC] / 3

        elif net["_options"]["fault"] == "LL":  # line-to-line fault 7.3 in standard
            '''
            Dividing IKSSC by √3 gives correct sequence currents for L–L faults.
            This matches the scaling applied in calc_ikss_to_g,
            where IKSSC is internally multiplied by √3/2 for this fault type.
            '''
            I1 = ppci_1["bus"][bus, IKSSC] /sqrt(3)
            I2 = -I1
            I0 = ppci_0["bus"][bus, IKSSC] # or I0 = np.zeros_like(I1)

        elif net["_options"]["fault"] == "LLG":  # double line-to-ground fault - no results are matching - 7.4 standard
            I1 = ppci_1["bus"][bus, IKSSC]
            I2 = ppci_2["bus"][bus, IKSSC]
            I0 = ppci_0["bus"][bus, IKSSC]

        I_phase = sequence_to_phase(np.array([I0, I1, I2], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase

        # Phase voltage (kV) for MVA calculation
        V_phase_kv = vn_sgen/ np.sqrt(3)
        # Apparent power per phase
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        # Total apparent power
        skss = skss_a_mva + skss_b_mva + skss_c_mva

        # --- Store result ---
        results.append({
            "bus": net.gen.bus.values[i],
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss_mva": skss,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
        })

    # --- Store in pandapower result table ---
    net.res_sgen_sc = pd.DataFrame(results)
    return net

In [ ]:
#VERSION 1.3 just the shell of what we need

def _get_sgen_results(net, ppci_0, ppci_1, ppci_2, bus_lookup):
    """
    Computes individual sgen short-circuit contributions (not summed per bus).
    Version 1.2 — based on pandapower currents.py logic, extended for per-sgen detail.
    """

    if "sgen" not in net or len(net.sgen) == 0:
        return

    fault = net._options["fault"]
    case = net._options["case"]
    fault_impedance = net._options.get("fault_impedance", 0)

    sgen_df = net.sgen.loc[net.sgen.in_service].copy()
    results = []

    # --- Global base ---
    S_base = net.sn_mva

    for idx, sgen in sgen_df.iterrows():
        bus = sgen.bus
        bus_ppc = bus
        vn_kv = net.bus.at[bus, "vn_kv"]
        baseI = vn_kv * np.sqrt(3) / S_base  # base current in kA
        sgen = net.sgen[net.sgen.in_service & net.sgen.current_source]

        # Get base system quantities
        base_mva = net.sn_mva
        base_kv = net.bus.vn_kv.loc[sgen.bus].values
        base_i = base_kv * np.sqrt(3) / base_mva  # base current per bus

        # Per-unit current for each sgen (same logic as pandapower)
        i_sgen_pu = np.where(
            sgen.active_current.values,
            (sgen.p_mw.values * sgen.scaling.values / base_mva * sgen.k.values),
            (sgen.sn_mva.values * sgen.scaling.values / base_mva * sgen.k.values)
        )

        # Convert to absolute amperes
        i_sgen_ka = np.abs(i_sgen_pu * base_i) / 1000  # convert A → kA

        # Equivalent impedances per sequence
        z0 = ppci_0["bus"][bus_ppc, R_EQUIV ] + 1j * ppci_0["bus"][bus_ppc, X_EQUIV]  # R_EQUIV + jX_EQUIV
        z1 = ppci_1["bus"][bus_ppc, R_EQUIV ] + 1j * ppci_1["bus"][bus_ppc, X_EQUIV]
        z2 = ppci_2["bus"][bus_ppc, R_EQUIV ] + 1j * ppci_2["bus"][bus_ppc, X_EQUIV]

        # --- Calculate base current contribution in p.u. ---
        I1_pu = (sgen.sn_mva / S_base) * sgen.k  # IEC Type A (current source)
        I0_pu = 0
        I2_pu = 0

        # --- Fault-type-dependent relationships (IEC 60909) ---
        if fault == "LLL":
            # 3-phase fault: only positive-sequence current
            I0_pu, I2_pu = 0, 0

        elif fault == "LG":
            # Line-to-earth: all sequences in series, same magnitude
            Z_eq = z0 + z1 + z2 + 3 * fault_impedance
            I1_pu = I1_pu * sgen.k * (z1 / Z_eq)
            I0_pu = I1_pu
            I2_pu = I1_pu

        elif fault == "LL":
            # Line-to-line: 1 and 2 sequences in opposition
            Z_eq = z1 + z2
            I1_pu = I1_pu * (z1 / Z_eq)
            I2_pu = -I1_pu
            I0_pu = 0

        elif fault == "LLG":
            # Double line-to-earth: complex impedance relationships
            denom = z2 * z0 + z1 * z0 + z1 * z2
            I1_pu = I1_pu * ((z1 * (z0 + z2)) / denom)
            I2_pu = I1_pu * ((z1 * z0) / denom)
            I0_pu = I1_pu * ((z1 * z2) / denom)

        else:
            continue  # unsupported fault type

        # --- Convert sequence currents to phase currents ---
        I_seq = np.array([I0_pu, I1_pu, I2_pu], dtype=complex)
        I_phase = sequence_to_phase(I_seq).squeeze()
        I_a, I_b, I_c = I_phase * baseI  # convert back to kA base

        # --- Apparent power contributions ---
        V_phase_kv = vn_kv / np.sqrt(3)
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss_total = skss_a_mva + skss_b_mva + skss_c_mva

        # --- Store results per sgen ---
        results.append({
            "sgen_index": idx,
            "bus": bus,
            "vn_kv": vn_kv,
            "fault": fault,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "phi_a_deg": np.angle(I_a, deg=True),
            "phi_b_deg": np.angle(I_b, deg=True),
            "phi_c_deg": np.angle(I_c, deg=True),
            "skss_mva": skss_total,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c))
        })

    # --- Return or store in net ---
    net.res_sgen_sc = pd.DataFrame(results)

In [ ]:
#Version 1.4 - extracting only static generator contrbutin - all faults works for one sgen
def _get_sgen_results(net, ppci_0, ppci_1, ppci_2, bus):
    if "sgen" not in net or len(net.sgen) == 0:
        return

    # --- Map pandapower to ppc indices ---
    bus_lookup = net["_pd2ppc_lookups"]["bus"]
    sgen_df = net.sgen.loc[net.sgen.in_service]
    results = []

    # --- Define helper function to compute single sgen current contribution ---
    def _single_current_source_current(net, ppci, bus_idx, sgen_idx, sequence=1):
        """
        Compute the short-circuit current contribution of a single sgen
        using pandapower's internal approach (based on _current_source_current).
        """
        sgen = net.sgen.loc[sgen_idx]
        if not sgen.in_service or not sgen.current_source:
            return 0

        baseI = ppci["internal"]["baseI"]
        bus_lookup = net["_pd2ppc_lookups"]["bus"]
        sgen_bus_ppc = bus_lookup[sgen.bus]
        fault_impedance = net._options["fault_impedance"]

        if net["_options"]["fault"] == "LL":
            fault_impedance /= 2

        # --- Compute injected current in p.u. ---
        if sgen.active_current:
            i_sgen_pu = (sgen.p_mw * sgen.scaling / net.sn_mva * sgen.k)
        else:
            i_sgen_pu = (sgen.sn_mva * sgen.scaling / net.sn_mva * sgen.k)

        if "current_angle_degree" in sgen:
            i_sgen_pu *= np.exp(1j * np.deg2rad(sgen.current_angle_degree))

        # --- Build injection vector ---
        i_inj_vec = np.zeros(ppci["bus"].shape[0], dtype=complex)
        i_inj_vec[sgen_bus_ppc] = i_sgen_pu

        # --- Propagate through Zbus or Ybus ---
        if net["_options"]["inverse_y"]:
            Zbus = ppci["internal"]["Zbus"].copy()
            diagZ = np.diag(Zbus).copy()
            diagZ[bus_idx] += fault_impedance
            i_kss_complex = 1 / diagZ * np.dot(Zbus, i_inj_vec)
        else:
            ybus_fact = ppci["internal"]["ybus_fact"]
            diagZ = np.diag(ppci["internal"]["Zbus"])
            diagZ[bus_idx] += fault_impedance
            i_kss_complex = ybus_fact(i_inj_vec) / diagZ

        return i_kss_complex[bus_idx] / baseI

    # --- Iterate over in-service sgens ---
    for i, s in sgen_df.iterrows():
        bus = s.bus
        bus_ppc = bus_lookup[bus]
        vn_sgen = net.bus.at[bus, "vn_kv"]

        # --- Compute per-sequence contributions ---
        ikssc_sgen = _single_current_source_current(net, ppci_1, bus_ppc, i, sequence=1)
        ikssc_1=ikssc_sgen[bus]

        z_equiv_0 = ppci_0["bus"][bus, R_EQUIV] + ppci_0["bus"][bus, X_EQUIV] * 1j
        z_equiv_1 = ppci_1["bus"][bus, R_EQUIV] + ppci_1["bus"][bus, X_EQUIV] * 1j
        z_equiv_2 = ppci_2["bus"][bus, R_EQUIV] + ppci_2["bus"][bus, X_EQUIV] * 1j


        if net["_options"]["fault"] == "LLL":
            I1 = ikssc_1
            I2 =np.zeros_like(I1)
            I0 =np.zeros_like(I1)

        elif net["_options"]["fault"] == "LG":
            factor = abs( z_equiv_1 / (z_equiv_1 + z_equiv_2 + z_equiv_0))
            I1 = factor * ikssc_1
            I2 = factor * ikssc_1
            I0 = factor * ikssc_1

        elif net["_options"]["fault"] == "LL":
            I1 = ikssc_1 / 2 #(I1=Eg/(Z1+Z2) and Eg=ikssc_1*Z1  and Z1==Z2)
            I2 = -I1
            I0 = 0

        elif net["_options"]["fault"] == "LLG":
             I1 = abs(ikssc_1 * ((z_equiv_1 * (z_equiv_0 + z_equiv_2)) / (z_equiv_2 * z_equiv_0 + z_equiv_1 * z_equiv_0 + z_equiv_1 * z_equiv_2)))
             I2 = -I1 * z_equiv_0 / (z_equiv_2 + z_equiv_0)
             I0 = -I1 * z_equiv_2 / (z_equiv_2 + z_equiv_0)


             # --- Convert from sequence to phase ---
        I_phase = sequence_to_phase(np.array([I0, I1, I2], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase

        # --- Calculate apparent power contributions ---
        V_phase_kv = vn_sgen / np.sqrt(3)
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva

        results.append({
            "bus": s.bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss_mva": skss,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
        })

    # --- Store results ---
    net.res_sgen_sc = pd.DataFrame(results)
    return net


In [ ]:
#version 1.5 -all fault swork for every combination - FOR iteration -not 100% explained yet

def _get_sgen_results(net, ppci_0, ppci_1, ppci_2, bus):
    if "sgen" not in net or len(net.sgen) == 0:
        return

    # Map pandapower to ppc indices
    bus_lookup = net["_pd2ppc_lookups"]["bus"]
    sgen_df = net.sgen.loc[net.sgen.in_service]
    results = []
    bus_idx=bus
    z_equiv_0 = ppci_0["bus"][bus_idx, R_EQUIV] + ppci_0["bus"][bus_idx, X_EQUIV] * 1j
    z_equiv_1 = ppci_1["bus"][bus_idx, R_EQUIV] + ppci_1["bus"][bus_idx, X_EQUIV] * 1j
    z_equiv_2 = ppci_2["bus"][bus_idx, R_EQUIV] + ppci_2["bus"][bus_idx, X_EQUIV] * 1j

    def _single_current_source_current(net, ppci, bus_idx, sgen_idx, sequence=1):
        """
        Calculates the short-circuit current contribution of a single static generator (sgen)
        to a fault at a specific bus (bus_idx).

        The idea:
        Each sgen injects a current at its own connection bus (sgen.bus).
        The network impedance matrix (Zbus or Ybus) is then used to calculate
        how that current propagates through the system and contributes to the
        fault current at the faulted bus (bus_idx).

        Parameters
        net : pandapower network
            The pandapower network object containing buses, lines, and generators.
        ppci : dict
            The internal ppc representation (for one sequence system).
            It contains Zbus, Ybus, and base quantities.
        bus_idx : int
            Index of the faulted bus in the ppc structure (where the short-circuit occurs).
        sgen_idx : int
            Index of the static generator in net.sgen table.
        sequence : int, optional
            The sequence system (0, 1, or 2). Default is 1 (positive sequence).

        Returns
        complex
            The contribution of this sgen to the fault current at bus_idx (in per-unit).
        """

        # Access the generator data
        sgen = net.sgen.loc[sgen_idx]

        # If the sgen is not active or not modeled as a current source → no contribution
        if not sgen.in_service or not sgen.current_source:
            return 0

        # Base current for per-unit scaling
        # This is typically Ibase = Sbase / (sqrt(3) * Vbase)
        baseI = ppci["internal"]["baseI"]

        # Map pandapower bus indices to internal ppc indices
        bus_lookup = net["_pd2ppc_lookups"]["bus"]
        sgen_bus_ppc = bus_lookup[sgen.bus]  # the bus where the generator is physically connected

        # Retrieve the fault impedance (if any)
        fault_impedance = net._options["fault_impedance"]

        # For line-to-line (LL) faults, the effective impedance per phase is halved
        if net["_options"]["fault"] == "LL":
            fault_impedance /= 2

        # Compute the injected current in per-unit
        # This is the current that the sgen will push into its bus under fault conditions
        if sgen.active_current:
            # If the generator defines its current from active power (P)
            i_sgen_pu = (sgen.p_mw * sgen.scaling / net.sn_mva * sgen.k)
        else:
            # If the generator defines its current from apparent power (S)
            i_sgen_pu = (sgen.sn_mva * sgen.scaling / net.sn_mva * sgen.k)

        # Apply current phase angle if defined
        # This represents the phase displacement of the injected current
        if "current_angle_degree" in sgen:
            i_sgen_pu *= np.exp(1j * np.deg2rad(sgen.current_angle_degree))

        # Build the injection vector
        # Each bus gets a current injection value
        # All buses are zero except the one where this sgen is connected
        i_inj_vec = np.zeros(ppci["bus"].shape[0], dtype=complex)
        i_inj_vec[sgen_bus_ppc] = i_sgen_pu

        # Propagate the current injection through the network
        # This step calculates how much of the injected current reaches the faulted bus
        # Using the Zbus matrix: V = Zbus * Iinj  ⇒  Ifault = V_fault / Zbus_diag(fault)
        # or the Ybus factorization if available

        if net["_options"]["inverse_y"]:
            # Case 1: Zbus matrix is available directly
            Zbus = ppci["internal"]["Zbus"].copy()
            diagZ = np.diag(Zbus).copy()
            diagZ[bus_idx] += fault_impedance  # add fault impedance to diagonal term

            # Compute voltage vector caused by this current injection
            V = np.dot(Zbus, i_inj_vec)

            # Convert to fault current contribution (Ohm’s law)
            i_kss_complex = V / diagZ

        else:
            # Case 2: If only Ybus factorization is available
            ybus_fact = ppci["internal"]["ybus_fact"]
            diagZ = np.diag(ppci["internal"]["Zbus"])
            diagZ[bus_idx] += fault_impedance
            # Compute voltage drop and divide by Z to get current at fault bus
            i_kss_complex = ybus_fact(i_inj_vec) / diagZ

        # Extract the contribution at the faulted bus
        # Only the current at bus_idx is relevant for total fault current calculation.
        i_fault_contribution = i_kss_complex[bus_idx] / baseI

        # Return the complex fault current contribution of this sgen
        return i_fault_contribution

    # Loop over all in-service static generators
    for i, s in sgen_df.iterrows():
        bus = s.bus
        bus_ppc = bus_lookup[bus]
        vn_sgen = net.bus.at[bus, "vn_kv"]

        # Compute positive sequence current contribution
        # This gives the current injected by this generator seen at the faulted bus.
        ikssc_sgen = _single_current_source_current(net, ppci_1, bus_idx, i, sequence=1)
        ikssc_1=ikssc_sgen[bus]

        if net["_options"]["fault"] == "LLL":
            #3-phase fault: only positive sequence current contributes
            I1 = ikssc_1
            I2 =np.zeros_like(I1)
            I0 =np.zeros_like(I1)

        elif net["_options"]["fault"] == "LG":
            # Single line-to-ground fault: all three sequence networks are in series
            # so current from LLL fault  is scaled by Z1 / (Z0 + Z1 + Z2) — implemented via “factor” below -EXPLAIN THIS TO HENDRIK => 7.5 IN STANDARD
            factor = abs( z_equiv_1 / (z_equiv_1 + z_equiv_2 + z_equiv_0))
            I1 = factor * ikssc_1
            I2 = factor * ikssc_1
            I0 = factor * ikssc_1

        elif net["_options"]["fault"] == "LL":
            #Line-to-line fault:positive and negative sequence networks are active and equal in magnitude but opposite in phase.
            #Pandapower computes IKSSC for LL faults using √3/2 scaling,so dividing by 2 yields correct magnitude PER ONE PHASE - 7.3 IN STANDARD
            I1 = ikssc_1 / 2 #(I1=Eg/(Z1+Z2) and Eg=ikssc_1*Z1  and Z1==Z2)
            I2 = -I1
            I0 = 0

        elif net["_options"]["fault"] == "LLG":
            #Double line-to-ground fault:all three sequence networks are active but connected in a more complex parallel form.
            #The sequence currents are derived from symmetrical component analysis - 7.4 IN STANDARD
             I1 = abs(ikssc_1 * ((z_equiv_1 * (z_equiv_0 + z_equiv_2)) / (z_equiv_2 * z_equiv_0 + z_equiv_1 * z_equiv_0 + z_equiv_1 * z_equiv_2)))
             I2 = -I1 * z_equiv_0 / (z_equiv_2 + z_equiv_0)
             I0 = -I1 * z_equiv_2 / (z_equiv_2 + z_equiv_0)


        # Convert from sequence to phase
        # Transform sequence domain (I0, I1, I2) into actual phase currents (Ia, Ib, Ic)
        I_phase = sequence_to_phase(np.array([I0, I1, I2], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase

        # Compute apparent power contribution (MVA) per phase
        V_phase_kv = vn_sgen / np.sqrt(3)
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva

        results.append({
            "bus": s.bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss_mva": skss,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
        })

    # Store results
    net.res_sgen_sc = pd.DataFrame(results)
    return net

In [ ]:
#Version 1.6 - vectorized but not fully understandable to be honest
def _get_sgen_results(net, ppci_0, ppci_1, ppci_2, bus):
    """
    Vectorized calculation of short-circuit contributions from all static generators.
    Each sgen contribution is calculated for the specific fault bus location.
    Results include NaN values for sgens that are not in service.
    """
    if "sgen" not in net or len(net.sgen) == 0:
        return

    # Get all sgens (not just in_service ones)
    sgen_df = net.sgen
    n_sgen = len(sgen_df)

    if n_sgen == 0:
        return

    # Map pandapower to ppc indices
    bus_lookup = net["_pd2ppc_lookups"]["bus"]

    # Get fault bus indices - can be array if multiple faults
    if isinstance(bus, (list, np.ndarray)):
        bus_fault_ppc = np.array([bus_lookup[b] for b in bus])
    else:
        bus_fault_ppc = np.array([bus_lookup[bus]])

    n_faults = len(bus_fault_ppc)

    # Get sgen bus indices in ppc coordinates
    sgen_buses_ppc = np.array([bus_lookup[b] for b in sgen_df.bus.values])

    # Get equivalent impedances at each fault bus for all sequences
    # Shape: (n_faults, 3) for [Z0, Z1, Z2]
    z_equiv = np.zeros((n_faults, 3), dtype=complex)
    for i, bf_idx in enumerate(bus_fault_ppc):
        z_equiv[i, 0] = ppci_0["bus"][bf_idx, R_EQUIV] + ppci_0["bus"][bf_idx, X_EQUIV] * 1j  # Z0
        z_equiv[i, 1] = ppci_1["bus"][bf_idx, R_EQUIV] + ppci_1["bus"][bf_idx, X_EQUIV] * 1j  # Z1
        z_equiv[i, 2] = ppci_2["bus"][bf_idx, R_EQUIV] + ppci_2["bus"][bf_idx, X_EQUIV] * 1j  # Z2

    # Fault impedance
    fault_impedance = net._options["fault_impedance"]
    if net["_options"]["fault"] == "LL":
        fault_impedance /= 2

    # Prepare masks for active sgens
    mask_in_service = sgen_df.in_service.values
    mask_current_source = sgen_df.current_source.values if 'current_source' in sgen_df.columns else np.ones(n_sgen,
                                                                                                            dtype=bool)
    active_mask = mask_in_service & mask_current_source

    # Calculate per-unit currents for all sgens
    i_sgen_pu = np.zeros(n_sgen, dtype=complex)
    mask_active = sgen_df.active_current.values if 'active_current' in sgen_df.columns else np.zeros(n_sgen, dtype=bool)

    # Active power based current
    if mask_active.any():
        i_sgen_pu[mask_active] = (sgen_df.loc[mask_active, 'p_mw'].values *
                                  sgen_df.loc[mask_active, 'scaling'].values /
                                  net.sn_mva *
                                  sgen_df.loc[mask_active, 'k'].values)

    # Apparent power based current
    if (~mask_active).any():
        i_sgen_pu[~mask_active] = (sgen_df.loc[~mask_active, 'sn_mva'].values *
                                   sgen_df.loc[~mask_active, 'scaling'].values /
                                   net.sn_mva *
                                   sgen_df.loc[~mask_active, 'k'].values)

    # Apply phase angle if exists
    if "current_angle_degree" in sgen_df.columns:
        angles = sgen_df.current_angle_degree.values
        i_sgen_pu *= np.exp(1j * np.deg2rad(angles))

    # Set currents to zero for inactive sgens
    i_sgen_pu[~active_mask] = 0

    # Calculate contributions for each fault location
    # Result shape: (n_faults, n_sgen) - fault current from each sgen to each fault bus
    n_buses = ppci_1["bus"].shape[0]
    ikssc_all = np.zeros((n_faults, n_sgen), dtype=complex)

    # Ensure baseI is properly handled
    baseI_val = ppci_1["internal"]["baseI"]
    if isinstance(baseI_val, np.ndarray):
        baseI_val = baseI_val[0] if baseI_val.size == 1 else baseI_val

    for fault_idx, bf_idx in enumerate(bus_fault_ppc):
        # Build injection matrix for this fault
        i_inj_matrix = np.zeros((n_buses, n_sgen), dtype=complex)

        for sgen_idx, (bus_ppc, current) in enumerate(zip(sgen_buses_ppc, i_sgen_pu)):
            i_inj_matrix[bus_ppc, sgen_idx] = current

        # Propagate currents through network
        if net["_options"]["inverse_y"]:
            Zbus = ppci_1["internal"]["Zbus"].copy()
            diagZ = np.diag(Zbus).copy()
            diagZ[bf_idx] += fault_impedance

            # V = Zbus @ i_inj (for all sgens at once)
            V = Zbus @ i_inj_matrix  # (n_buses x n_sgen)

            # Current at each bus for each sgen
            i_kss_complex = V / diagZ[:, np.newaxis]
        else:
            ybus_fact = ppci_1["internal"]["ybus_fact"]
            diagZ = np.diag(ppci_1["internal"]["Zbus"]).copy()
            diagZ[bf_idx] += fault_impedance

            # Apply ybus_fact to each column (each sgen)
            i_kss_complex = np.zeros((n_buses, n_sgen), dtype=complex)
            for sgen_idx in range(n_sgen):
                i_kss_complex[:, sgen_idx] = ybus_fact(i_inj_matrix[:, sgen_idx]) / diagZ

        # Extract fault current contributions at this fault bus for all sgens
        # i_kss_complex shape: (n_buses, n_sgen)
        # Extract row corresponding to fault bus
        fault_currents = i_kss_complex[bf_idx, :]  # Shape: (n_sgen,)

        # Normalize by base current
        if isinstance(baseI_val, np.ndarray) and baseI_val.ndim > 0 and len(baseI_val) > 1:
            ikssc_all[fault_idx, :] = fault_currents / baseI_val[bf_idx]
        else:
            base_i_scalar = float(baseI_val) if np.isscalar(baseI_val) else float(
                baseI_val.item() if baseI_val.size == 1 else baseI_val)
            ikssc_all[fault_idx, :] = fault_currents / base_i_scalar

    # Now process each sgen - each may contribute to different fault location
    # Assuming single fault bus (most common case)
    if n_faults == 1:
        ikssc_sgen = ikssc_all[0, :]  # Shape: (n_sgen,)
        z_equiv_0 = z_equiv[0, 0]
        z_equiv_1 = z_equiv[0, 1]
        z_equiv_2 = z_equiv[0, 2]
    else:
        # If multiple faults, need to match each sgen to its fault bus
        # This requires additional logic - for now assume single fault
        raise NotImplementedError("Multiple fault buses not yet implemented")

    # Initialize result arrays
    I_a = np.full(n_sgen, np.nan, dtype=complex)
    I_b = np.full(n_sgen, np.nan, dtype=complex)
    I_c = np.full(n_sgen, np.nan, dtype=complex)

    # Calculate sequence currents based on fault type (vectorized)
    if net["_options"]["fault"] == "LLL":
        I1 = ikssc_sgen
        I2 = np.zeros_like(I1)
        I0 = np.zeros_like(I1)

    elif net["_options"]["fault"] == "LG":
        factor = abs(z_equiv_1 / (z_equiv_1 + z_equiv_2 + z_equiv_0))
        I1 = factor * ikssc_sgen
        I2 = factor * ikssc_sgen
        I0 = factor * ikssc_sgen

    elif net["_options"]["fault"] == "LL":
        I1 = ikssc_sgen / 2
        I2 = -I1
        I0 = np.zeros_like(I1)

    elif net["_options"]["fault"] == "LLG":
        I1 = abs(ikssc_sgen * ((z_equiv_1 * (z_equiv_0 + z_equiv_2)) /
                               (z_equiv_2 * z_equiv_0 + z_equiv_1 * z_equiv_0 + z_equiv_1 * z_equiv_2)))
        I2 = -I1 * z_equiv_0 / (z_equiv_2 + z_equiv_0)
        I0 = -I1 * z_equiv_2 / (z_equiv_2 + z_equiv_0)

    # Vectorized sequence to phase transformation
    # Stack all sequence currents: shape (3, n_sgen)
    I_seq_all = np.vstack([I0, I1, I2])

    # Transform for active sgens only
    for idx in range(n_sgen):
        if active_mask[idx]:
            I_seq = I_seq_all[:, idx]
            I_phase = sequence_to_phase(I_seq).squeeze()
            I_a[idx], I_b[idx], I_c[idx] = I_phase

    # Get voltage levels for all sgen buses
    # Each sgen connected to its own bus, so we look up vn_kv for each sgen.bus
    vn_sgen = np.array([net.bus.at[b, "vn_kv"] for b in sgen_df.bus.values])
    V_phase_kv = vn_sgen / np.sqrt(3)

    # Calculate apparent power contributions (vectorized)
    skss_a_mva = V_phase_kv * np.abs(I_a)
    skss_b_mva = V_phase_kv * np.abs(I_b)
    skss_c_mva = V_phase_kv * np.abs(I_c)
    skss = skss_a_mva + skss_b_mva + skss_c_mva

    # Build results dataframe
    results_dict = {
        "bus": sgen_df.bus.values,
        "ikss_a_ka": np.abs(I_a),
        "ikss_b_ka": np.abs(I_b),
        "ikss_c_ka": np.abs(I_c),
        "ikss_a_degree": np.angle(I_a, deg=True),
        "ikss_b_degree": np.angle(I_b, deg=True),
        "ikss_c_degree": np.angle(I_c, deg=True),
        "skss_mva": skss,
        "skss_a_mva": skss_a_mva,
        "skss_b_mva": skss_b_mva,
        "skss_c_mva": skss_c_mva,
        "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
        "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
        "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
        "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
        "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
        "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
    }

    net.res_sgen_sc = pd.DataFrame(results_dict, index=sgen_df.index)
    return net

In [ ]:
#VERSION  2.0 - VECTORIZED AND FULLY UNDERSTANDABLE

def _get_sgen_results(net, ppci_0, ppci_1, ppci_2, bus):

    # NOTE: This function is implemented based on _current_source_current fucntion from pandapower/shortcircuit/currents.py
    # Early exit if no static generators exist in the network
    if "sgen" not in net or len(net.sgen) == 0:
        return

    # Get all sgens (including those not in service) to maintain consistent indexing
    sgen_df = net.sgen
    n_sgen = len(sgen_df)

    if n_sgen == 0:
        return


    # STEP 1: Map pandapower bus indices to internal ppc indices
    # This is necessary because internal calculations use ppc indexing
    bus_lookup = net["_pd2ppc_lookups"]["bus"]


    bus_fault_ppc = np.array([bus_lookup[bus]])

    n_faults = len(bus_fault_ppc)

    # Get ppc indices for all sgen connection buses
    # These are the buses where generators inject their currents
    sgen_buses_ppc = np.array([bus_lookup[b] for b in sgen_df.bus.values])

    # STEP 2: Extract equivalent impedances at fault location(s)

    # Get equivalent impedances at each fault bus for all sequence networks
    # These impedances represent the Thévenin equivalent of the network as seen
    # from the fault location, crucial for calculating fault currents
    # Shape: (n_faults, 3) for [Z0, Z1, Z2]
    z_equiv = np.zeros((n_faults, 3), dtype=complex)
    for i, bf_idx in enumerate(bus_fault_ppc):
        # Zero sequence equivalent impedance (for ground faults)
        z_equiv[i, 0] = ppci_0["bus"][bf_idx, R_EQUIV] + ppci_0["bus"][bf_idx, X_EQUIV] * 1j
        # Positive sequence equivalent impedance (always present)
        z_equiv[i, 1] = ppci_1["bus"][bf_idx, R_EQUIV] + ppci_1["bus"][bf_idx, X_EQUIV] * 1j
        # Negative sequence equivalent impedance (for unbalanced faults)
        z_equiv[i, 2] = ppci_2["bus"][bf_idx, R_EQUIV] + ppci_2["bus"][bf_idx, X_EQUIV] * 1j


    # STEP 3: Setup fault impedance
    # Fault impedance represents any additional impedance at the fault point
    # (e.g., arc resistance, grounding resistance)
    fault_impedance = net._options["fault_impedance"]

    # For line-to-line faults, the effective impedance per phase is halved
    # because two phases are involved in parallel
    if net["_options"]["fault"] == "LL":
        fault_impedance /= 2


    # STEP 4: Identify active static generators and calculate their injection currents

    # Create masks to identify which sgens are active and should contribute
    mask_in_service = sgen_df.in_service.values
    mask_current_source = sgen_df.current_source.values if 'current_source' in sgen_df.columns else np.ones(n_sgen, dtype=bool)
    active_mask = mask_in_service & mask_current_source

    # Initialize per-unit current injections for all sgens
    i_sgen_pu = np.zeros(n_sgen, dtype=complex)

    # Determine if current is based on active power (P) or apparent power (S)
    mask_active = sgen_df.active_current.values if 'active_current' in sgen_df.columns else np.zeros(n_sgen, dtype=bool)

    # Calculate injection currents based on active power (P)
    # I_pu = (P_MW * scaling / S_base) * k_factor
    # where k_factor accounts for voltage dependency of the source
    if mask_active.any():
        i_sgen_pu[mask_active] = (sgen_df.loc[mask_active, 'p_mw'].values *
                                  sgen_df.loc[mask_active, 'scaling'].values /
                                  net.sn_mva *
                                  sgen_df.loc[mask_active, 'k'].values)

    # Calculate injection currents based on apparent power (S)
    # I_pu = (S_MVA * scaling / S_base) * k_factor
    if (~mask_active).any():
        i_sgen_pu[~mask_active] = (sgen_df.loc[~mask_active, 'sn_mva'].values *
                                   sgen_df.loc[~mask_active, 'scaling'].values /
                                   net.sn_mva *
                                   sgen_df.loc[~mask_active, 'k'].values)

    # Apply phase angle to injection current if specified
    # This represents the phase displacement of the injected current relative to voltage
    if "current_angle_degree" in sgen_df.columns:
        angles = sgen_df.current_angle_degree.values
        i_sgen_pu *= np.exp(1j * np.deg2rad(angles))

    # Set currents to zero for inactive sgens (but keep their array position)
    # This ensures NaN values in final results for inactive generators
    i_sgen_pu[~active_mask] = 0

    # STEP 5: Propagate currents through network using superposition

    # Calculate fault current contributions for each fault location
    # Result shape: (n_faults, n_sgen) - each sgen's contribution to each fault
    n_buses = ppci_1["bus"].shape[0]
    ikssc_all = np.zeros((n_faults, n_sgen), dtype=complex)

    # Extract base current for per-unit to physical unit conversion
    # baseI = S_base / (sqrt(3) * V_base)
    baseI_val = ppci_1["internal"]["baseI"]

    # Process each fault location
    for fault_idx, bf_idx in enumerate(bus_fault_ppc):

        # STEP 5a: Build current injection matrix

        # Create matrix where each column represents one sgen's injection
        # Rows represent buses - only the connection bus of each sgen is non-zero
        i_inj_matrix = np.zeros((n_buses, n_sgen), dtype=complex)

        for sgen_idx, (bus_ppc, current) in enumerate(zip(sgen_buses_ppc, i_sgen_pu)):
            i_inj_matrix[bus_ppc, sgen_idx] = current

        # STEP 5b: Network propagation using impedance/admittance matrix

        # This process shows how injected currents propagate through the network
        # Two methods available: direct Zbus or Ybus factorization

        if net["_options"]["inverse_y"]:
            # Method 1: Direct impedance matrix (Zbus) approach
            # V = Zbus * I_inj  → more intuitive but memory intensive

            Zbus = ppci_1["internal"]["Zbus"].copy()
            diagZ = np.diag(Zbus).copy()

            # Add fault impedance to the diagonal element at fault bus
            # This represents additional impedance at the fault point
            diagZ[bf_idx] += fault_impedance

            # Calculate voltage drop due to current injection: V = Z * I
            # Matrix multiplication handles all sgens simultaneously
            V = Zbus @ i_inj_matrix  # Shape: (n_buses, n_sgen)

            # Convert voltage drop to current: I = V / Z
            # Each column gives the current distribution from one sgen
            i_kss_complex = V / diagZ[:, np.newaxis]
        else:
            # Method 2: Admittance matrix factorization (Ybus) approach
            # Solve: Y * V = I_inj → more efficient for large systems

            ybus_fact = ppci_1["internal"]["ybus_fact"]  # Pre-factored Ybus
            diagZ = np.diag(ppci_1["internal"]["Zbus"]).copy()
            diagZ[bf_idx] += fault_impedance

            # Apply factorization to each sgen injection separately
            i_kss_complex = np.zeros((n_buses, n_sgen), dtype=complex)
            for sgen_idx in range(n_sgen):
                # Solve linear system and convert to current
                i_kss_complex[:, sgen_idx] = ybus_fact(i_inj_matrix[:, sgen_idx]) / diagZ


        # STEP 5c: Extract fault current at the faulted bus

        # From the full current distribution, extract only the current at fault bus
        fault_currents = i_kss_complex[bf_idx, :]  # Shape: (n_sgen,)

        # Convert from per-unit to physical units (kA)
        # Handle both scalar and array baseI values
        if isinstance(baseI_val, np.ndarray) and baseI_val.ndim > 0 and len(baseI_val) > 1:
            # baseI is per-bus array
            ikssc_all[fault_idx, :] = fault_currents / baseI_val[bf_idx]
        else:
            # baseI is scalar
            base_i_scalar = float(baseI_val) if np.isscalar(baseI_val) else float(
                baseI_val.item() if baseI_val.size == 1 else baseI_val)
            ikssc_all[fault_idx, :] = fault_currents / base_i_scalar


    # STEP 6: Extract results for single fault case

    # For now, handle only single fault location (most common case)
    if n_faults == 1:
        ikssc_sgen = ikssc_all[0, :]  # Shape: (n_sgen,)
        z_equiv_0 = z_equiv[0, 0]  # Zero sequence impedance
        z_equiv_1 = z_equiv[0, 1]  # Positive sequence impedance
        z_equiv_2 = z_equiv[0, 2]  # Negative sequence impedance
    else:
        # Multiple faults require mapping each sgen to its respective fault bus
        raise NotImplementedError("Multiple fault buses not yet implemented")


    # STEP 7: Calculate sequence currents based on fault type

    # Initialize result arrays with NaN (will remain NaN for inactive sgens)
    I_a = np.full(n_sgen, np.nan, dtype=complex)
    I_b = np.full(n_sgen, np.nan, dtype=complex)
    I_c = np.full(n_sgen, np.nan, dtype=complex)

    # Calculate symmetrical components (I0, I1, I2) based on fault type
    # These follow IEC 60909 standard equations for different fault configurations

    if net["_options"]["fault"] == "LLL":
        # Three-phase balanced fault (Chapter 4.2 of IEC 60909)
        # Only positive sequence network is active
        # I1 = Ikss (calculated from positive sequence network)
        # I2 = I0 = 0 (no unbalanced components)
        I1 = ikssc_sgen
        I2 = np.zeros_like(I1)
        I0 = np.zeros_like(I1)

    elif net["_options"]["fault"] == "LG":
        # Single line-to-ground fault (Chapter 4.3.2 of IEC 60909)
        # All three sequence networks connected in series
        # I0 = I1 = I2 = E / (Z0 + Z1 + Z2)
        # The factor accounts for voltage division across sequence impedances
        factor = abs(z_equiv_1 / (z_equiv_1 + z_equiv_2 + z_equiv_0))
        I1 = factor * ikssc_sgen
        I2 = factor * ikssc_sgen
        I0 = factor * ikssc_sgen

    elif net["_options"]["fault"] == "LL":
        # Line-to-line fault (Chapter 4.3.3 of IEC 60909)
        # Positive and negative sequence networks in series
        # I1 = -I2 = E / (Z1 + Z2)
        # I0 = 0 (no ground path)
        # Division by 2 accounts for √3/2 scaling in pandapower's LL calculation
        I1 = ikssc_sgen / 2
        I2 = -I1
        I0 = np.zeros_like(I1)

    elif net["_options"]["fault"] == "LLG":
        # Double line-to-ground fault (Chapter 4.3.4 of IEC 60909)
        # All three sequence networks active in complex parallel configuration
        # I1 = E / (Z1 + Z2||Z0)
        # I2 and I0 follow from current divider principle
        I1 = abs(ikssc_sgen * ((z_equiv_1 * (z_equiv_0 + z_equiv_2)) /
                               (z_equiv_2 * z_equiv_0 + z_equiv_1 * z_equiv_0 + z_equiv_1 * z_equiv_2)))
        I2 = -I1 * z_equiv_0 / (z_equiv_2 + z_equiv_0)
        I0 = -I1 * z_equiv_2 / (z_equiv_2 + z_equiv_0)


    # STEP 8: Transform symmetrical components to phase quantities

    # Stack all sequence currents into matrix form
    # Shape: (3, n_sgen) where rows are [I0, I1, I2]
    I_seq_all = np.vstack([I0, I1, I2])


    # This converts sequence domain to physical phase currents
    for idx in range(n_sgen):
        if active_mask[idx]:
            I_seq = I_seq_all[:, idx]
            I_phase = sequence_to_phase(I_seq).squeeze()
            I_a[idx], I_b[idx], I_c[idx] = I_phase


    # STEP 9: Calculate apparent power contributions

    # Get voltage levels for all sgen connection buses
    # Each sgen may be connected to a bus with different voltage level
    vn_sgen = np.array([net.bus.at[b, "vn_kv"] for b in sgen_df.bus.values])

    # Convert line-to-line voltage to line-to-neutral (phase) voltage
    # V_phase = V_line / √3
    # This is needed because phase currents are line-to-neutral quantities
    V_phase_kv = vn_sgen / np.sqrt(3)

    # Calculate apparent power per phase: S = V * I
    # Units: kV * kA = MVA
    skss_a_mva = V_phase_kv * np.abs(I_a)
    skss_b_mva = V_phase_kv * np.abs(I_b)
    skss_c_mva = V_phase_kv * np.abs(I_c)

    # Total three-phase apparent power
    skss = skss_a_mva + skss_b_mva + skss_c_mva

    # STEP 10: Assemble results dataframe

    # Build comprehensive results dictionary with all calculated quantities
    results_dict = {
        # Bus identification
        "bus": sgen_df.bus.values,

        # Current magnitudes per phase (kA)
        "ikss_a_ka": np.abs(I_a),
        "ikss_b_ka": np.abs(I_b),
        "ikss_c_ka": np.abs(I_c),

        # Current angles per phase (degrees)
        "ikss_a_degree": np.angle(I_a, deg=True),
        "ikss_b_degree": np.angle(I_b, deg=True),
        "ikss_c_degree": np.angle(I_c, deg=True),

        # Apparent power quantities (MVA)
        "skss_mva": skss,  # Total three-phase
        "skss_a_mva": skss_a_mva,  # Phase A
        "skss_b_mva": skss_b_mva,  # Phase B
        "skss_c_mva": skss_c_mva,  # Phase C

        # Active power per phase (MW): P = S * cos(φ)
        "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
        "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),  # Reactive power (MVAR)
        "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
        "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
        "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
        "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
    }

    # Create results DataFrame preserving original sgen indices
    # Inactive sgens will have NaN values for all calculated quantities
    net.res_sgen_sc = pd.DataFrame(results_dict, index=sgen_df.index)

    return net